In [5]:
import httpx
import trafilatura
import json
import time

SEED_URLS = [
    "https://en.wikipedia.org/wiki/Stanley_Kubrick",
    "https://en.wikipedia.org/wiki/Christopher_Nolan",
    "https://en.wikipedia.org/wiki/Martin_Scorsese",
    "https://en.wikipedia.org/wiki/2001:_A_Space_Odyssey",
    "https://en.wikipedia.org/wiki/The_Godfather",
    "https://en.wikipedia.org/wiki/Pulp_Fiction",
    "https://en.wikipedia.org/wiki/Inception",
    "https://en.wikipedia.org/wiki/Akira_Kurosawa",
]

# Wikipedia requires a descriptive User-Agent — be transparent about who you are
HEADERS = {
    "User-Agent": "CinemaKnowledgeGraphBot/1.0 (university lab project; contact@example.com)"
}

def is_useful(text: str, min_words: int = 500) -> bool:
    return len(text.split()) >= min_words

def fetch_and_clean(url: str) -> dict | None:
    try:
        response = httpx.get(url, timeout=15, follow_redirects=True, headers=HEADERS)
        response.raise_for_status()

        text = trafilatura.extract(response.text, include_comments=False, include_tables=False)

        if text and is_useful(text):
            return {
                "url": url,
                "word_count": len(text.split()),
                "text": text
            }
        else:
            print(f"[SKIPPED] {url} — not enough content.")
            return None

    except Exception as e:
        print(f"[ERROR] {url}: {e}")
        return None

with open("crawler_output.jsonl", "w", encoding="utf-8") as f:
    for url in SEED_URLS:
        result = fetch_and_clean(url)
        if result:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            print(f"[OK] {url} — {result['word_count']} words")
        time.sleep(1)

print("Done. Output saved to crawler_output.jsonl")

[OK] https://en.wikipedia.org/wiki/Stanley_Kubrick — 22846 words
[OK] https://en.wikipedia.org/wiki/Christopher_Nolan — 13635 words
[OK] https://en.wikipedia.org/wiki/Martin_Scorsese — 20156 words
[OK] https://en.wikipedia.org/wiki/2001:_A_Space_Odyssey — 19721 words
[OK] https://en.wikipedia.org/wiki/The_Godfather — 16840 words
[OK] https://en.wikipedia.org/wiki/Pulp_Fiction — 19868 words
[OK] https://en.wikipedia.org/wiki/Inception — 14055 words
[OK] https://en.wikipedia.org/wiki/Akira_Kurosawa — 14321 words
Done. Output saved to crawler_output.jsonl


In [3]:
import spacy
import pandas as pd
import json

# Load the transformer-based model
nlp = spacy.load("en_core_web_trf")

# Entity labels relevant to cinema
TARGET_LABELS = {"PERSON", "ORG", "GPE", "DATE", "WORK_OF_ART"}

records = []

with open("crawler_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        url = entry["url"]
        doc = nlp(entry["text"])

        for ent in doc.ents:
            if ent.label_ in TARGET_LABELS:
                records.append({
                    "entity": ent.text,
                    "label": ent.label_,
                    "source_url": url
                })

df = pd.DataFrame(records).drop_duplicates()
df.to_csv("extracted_knowledge.csv", index=False)
print(df["label"].value_counts())
print(f"\nTotal unique entities: {len(df)}")

label
PERSON         3739
DATE           3703
WORK_OF_ART    1984
ORG            1708
GPE             375
Name: count, dtype: int64

Total unique entities: 11509


In [4]:
import spacy
import json

nlp = spacy.load("en_core_web_trf")
TARGET_LABELS = {"PERSON", "ORG", "GPE", "WORK_OF_ART"}

def extract_relations(doc):
    """Find entity pairs in the same sentence connected via a verb."""
    triples = []
    for sent in doc.sents:
        ents = [e for e in sent.ents if e.label_ in TARGET_LABELS]
        if len(ents) < 2:
            continue

        for token in sent:
            # Look for a verb that connects a subject and an object
            if token.pos_ == "VERB":
                subj = [c for c in token.children if c.dep_ == "nsubj"]
                obj  = [c for c in token.children if c.dep_ in ("dobj", "pobj", "attr")]

                if subj and obj:
                    triples.append({
                        "subject": subj[0].text,
                        "relation": token.lemma_,
                        "object": obj[0].text,
                        "sentence": sent.text.strip()
                    })
    return triples

all_triples = []

with open("crawler_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        doc = nlp(entry["text"])
        triples = extract_relations(doc)
        all_triples.extend(triples)

# Preview
for t in all_triples[:10]:
    print(f"({t['subject']}) --[{t['relation']}]--> ({t['object']})")
    print(f"  ↳ \"{t['sentence'][:100]}...\"")
    print()

/usr/local/lib/python3.12/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_trf' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


(Kubrick) --[teach]--> (producing)
  ↳ "Born in New York City, Kubrick taught himself film producing and directing after graduating from hig..."

(Kubrick) --[leave]--> (States)
  ↳ "In 1961, Kubrick left the United States and settled in England...."

(film) --[earn]--> (Award)
  ↳ "The scientific realism and innovative special effects in his science fiction epic 2001: A Space Odys..."

(Kubrick) --[withdraw]--> (which)
  ↳ "While many of Kubrick's films were controversial and initially received mixed reviews upon release—p..."

(Kubrick) --[obtain]--> (lenses)
  ↳ "For the 18th-century period film Barry Lyndon (1975), Kubrick obtained lenses developed by Carl Zeis..."

(NASA) --[film]--> (scenes)
  ↳ "For the 18th-century period film Barry Lyndon (1975), Kubrick obtained lenses developed by Carl Zeis..."

(he) --[become]--> (one)
  ↳ "With the horror film The Shining (1980), he became one of the first directors to make use of a Stead..."

(he) --[marry]--> (mother)
  ↳ "Jack, of Polis